## v0.1

In [1]:
# CANDIDATES="spotify_candidates_v0.1"
# INITIAL_IDS="initial_ids_v0.1"
# Q_JUDGE_PROMPT="q_judge_v0.1"
# Q_EXTRACT_PROMPT="q_extraction_v0.1"
# A_EXTRACT_PROMPT="a_extraction_v0.1"

CANDIDATES = "spotify_candidates_v0.1"
INITIAL_IDS = "initial_ids_v0.1"
Q_JUDGE_PROMPT = "q_judge_v0.1"
Q_EXTRACT_PROMPT = "q_extraction_v0.1"
A_EXTRACT_PROMPT = "a_extraction_v0.1"
MODEL = "gpt-4o-mini-2024-07-18"

MIN_MATCH_RATE_Q = 0.9
MIN_MATCH_RATE_A = 0.9
MAX_A_LENGTH = 500 # words

OUTFILE = "../output/pipeline_results_v0.1.tsv"

### Run

In [2]:
import pandas as pd

input_file = f"../output/data/{CANDIDATES}.tsv"
q_extraction_file = f"../output/q_extraction/parsed/{MODEL}/{Q_EXTRACT_PROMPT}/{INITIAL_IDS}.tsv"
quality_eval_file = f"../output/q_quality/parsed/{MODEL}/{Q_JUDGE_PROMPT}/{INITIAL_IDS}.tsv"
a_extraction_file = f"../output/a_extraction/parsed/{MODEL}/{Q_JUDGE_PROMPT}/{A_EXTRACT_PROMPT}/{INITIAL_IDS}.tsv"

df_input = pd.read_csv(input_file, sep="\t")
df_q_extraction = pd.read_csv(q_extraction_file, sep="\t")
df_quality = pd.read_csv(quality_eval_file, sep="\t")
df_a_extraction = pd.read_csv(a_extraction_file, sep="\t")

In [3]:
df_full = (
    df_q_extraction
        .merge(df_quality, on="podcast_id", how="left")
        .merge(df_a_extraction, on="podcast_id", how="left")
        .merge(df_input, on="podcast_id", how="left")
)

In [4]:
df_full.columns

Index(['podcast_id', 'question', 'q_verbatim', 'match_rate_q', 'q_span',
       'reasoning', 'info_seeking', 'info_seeking_logprob', 'self_contained',
       'self_contained_logprob', 'response', 'answer', 'a_verbatim',
       'match_rate_a', 'a_span', 'text'],
      dtype='object')

In [5]:
### FILTERING:

df_keep = df_full.copy()
print(f"{df_keep.shape[0]} -- Start")

# See where q is not nan:
mask_ = df_keep['question'].notnull()
df_keep = df_keep[mask_]
print(f"{df_keep.shape[0]} -- Found question")

# See where match_rate_q > MIN_MATCH_RATE_Q:
mask_ = df_keep['match_rate_q'] > MIN_MATCH_RATE_Q
df_keep = df_keep[mask_]
print(f"{df_keep.shape[0]} -- Match rate Q > {MIN_MATCH_RATE_Q}")

# See where 'info_seeking'==yes, 'self_contained'==yes:
mask_ = (df_full['info_seeking'] == 'yes') & (df_full['self_contained'] == 'yes')
df_keep = df_full[mask_].copy()
print(f"{df_keep.shape[0]} -- Good Q")

# Where answer is not nan:
mask_ = df_keep['answer'].notnull()
df_keep = df_keep[mask_]
print(f"{df_keep.shape[0]} -- Found answer")

# Where answer is after the question:
df_keep[["a_span", "q_span"]] = df_keep[["a_span", "q_span"]].map(lambda x: eval(x))
mask_ = df_keep.apply(lambda x: x['a_span'][0] > x['q_span'][1], axis=1)
df_keep = df_keep[mask_]
print(f"{df_keep.shape[0]} -- Answer after Q")

# Where answer is not very long:
mask_ = df_keep['a_verbatim'].str.count(" ") < MAX_A_LENGTH
df_keep = df_keep[mask_]
print(f"{df_keep.shape[0]} -- Answer not too long")

# See where match_rate_a > MIN_MATCH_RATE_A:
mask_ = df_keep['match_rate_a'] > MIN_MATCH_RATE_A
df_keep = df_keep[mask_]
print(f"{df_keep.shape[0]} -- Match rate > {MIN_MATCH_RATE_A}")

# Where the Q appears just once in the original text:
mask_ = df_keep.apply(lambda row: row['text'].lower().count(row['q_verbatim'].lower()) > 1, axis=1)
df_keep = df_keep[~mask_]
print(f"{df_keep.shape[0]} -- Q appears once")

# # Where the Q is not a subset of the A (is already implied in answer after Q):
# mask_good_qa = df_tmp.apply(lambda x: x['question'].lower().strip() not in x['answer'].lower(), axis=1)
# df_tmp = df_tmp[mask_good_qa]
# print(df_tmp.shape)

60 -- Start
47 -- Found question
34 -- Match rate Q > 0.9
13 -- Good Q
13 -- Found answer
9 -- Answer after Q
8 -- Answer not too long
7 -- Match rate > 0.9
6 -- Q appears once


In [6]:
# See some examples:
pd.set_option('display.max_colwidth', 200)
df_keep[["question", "a_verbatim", "match_rate_q", "match_rate_a"]].sort_values("match_rate_a", ascending=False).head(20)

,question,a_verbatim,match_rate_q,match_rate_a
10,What are the advantages of staying or returning to Real Madrid?,What is clear is I want to play and not be sat on the bench,1.0,1.000000
44,"""Can guys and girls be friends when one of them catches feelings?""",no no no i'm in this predicament like kind of right now like the guy that i'm talking to kind of goes with the fuck what do you think yeah um i'm kind of talking to this guy right now and he's lik...,1.0,1.000000
58,"""Could you give us just a high level overview of a mining pool, what it is and what it does?""","Sure, sure, sure. So a mining pool basically is the only way nowadays that you can mine Bitcoin and major cryptos because of the difficulty. So the more hash power that is connected into the Bitco...",1.0,1.000000
50,"Is there anything cooler than a billion, though?","Well, what I find out is that a trillion is actually better than that",1.0,1.000000
59,What is the gut brain about?,"Well, the gut brain is, you know, we have so many more messages come from the guts to the rest of the body than even the brain sends out. You know, if you think about your gut as your biggest orga...",1.0,1.000000
21,"""Do you have any comments or advice regarding future pre-med applicants who either come from business or marketing or just kind of is transitioning right now?""","i would say really take time to reflect and decide, see why you want to do it. Are you just bored with your current job? Are you trying different jobs? Is it like a longer term life issue where y...",1.0,0.963855


In [9]:
# NOTE here I send the output to a text file and manually select the good IDs:
df_tmp = df_keep[["podcast_id", "q_verbatim", "a_verbatim", "match_rate_a"]].sort_values("match_rate_a", ascending=False)

df_tmp.to_csv(f"../output/tmp.tsv", sep="\t", index=False)
for i, row in df_tmp.iterrows():
    print(f"{row['podcast_id']}\t{row['q_verbatim']}\t{row['a_verbatim']}")

podcasts-audio/5/9/show_59h8QSNa9n0SsLDAdH2GAA/4eNkaKFM0Ex4dbiExiuHlX	What are the advantages of staying or returning to Real Madrid	What is clear is I want to play and not be sat on the bench
podcasts-audio/3/H/show_3hjcWPBEicA1JnECxejrIZ/1IGmgW2ussPLS7x3l7JtAC	can guys and girls be friends when one of them catches feelings	no no no i'm in this predicament like kind of right now like the guy that i'm talking to kind of goes with the fuck what do you think yeah um i'm kind of talking to this guy right now and he's like he's brought it to my attention that he has his best friend's a girl and you know she's made it a point that very clear that she has feelings for him so the thing in this situation is like it's just kind of tricky but the answer is just no like don't get yourself in that position because you're never gonna fucking win yeah it's his best friend you're not gonna fucking get it past him honestly if they're close enough like they're gonna end up together at some point so and

In [10]:
# The manually selected IDs:
good_ids = [
    "podcasts-audio/5/9/show_59h8QSNa9n0SsLDAdH2GAA/4eNkaKFM0Ex4dbiExiuHlX",
    "podcasts-audio/3/H/show_3hjcWPBEicA1JnECxejrIZ/1IGmgW2ussPLS7x3l7JtAC",
    "podcasts-audio/2/C/show_2CLetGT20MFsHqfeBN3fYl/3EEivrpUk8v87BNaKjjXxz",
    "podcasts-audio/5/J/show_5JFugv0GD68SZC0Cuh1wXi/4eM1aRAeEDVZzxPv4Ko5DQ",
    "podcasts-audio/4/K/show_4kvJHO5LLxyfxbGPmAk2yQ/55kYxbeaCDVpnqwIe2Q225",
    "podcasts-audio/3/X/show_3XVVceNECAwk8xvSSYdgIO/7itwLRPH0t1ai26BTdztAA",
]

In [14]:
### SAVE
df_to_save = (
    df_keep
    [["podcast_id", "question", "q_verbatim", "match_rate_q", "q_span",
      "answer", "a_verbatim", "match_rate_a", "a_span"]]
)
df_to_save = df_to_save[df_to_save["podcast_id"].isin(good_ids)]

# strip quotation marks from the strings:
str_cols = ["question", "q_verbatim", "answer", "a_verbatim"]
for col in str_cols:
    df_to_save[col] = df_to_save[col].str.strip("\"").str.strip()
df_to_save.to_csv(OUTFILE, sep="\t", index=False)

print(f"Saved {df_to_save.shape[0]} rows to {OUTFILE}")
df_to_save.head(2)

Saved 6 rows to ../output/pipeline_results_v0.1.tsv


,podcast_id,question,q_verbatim,match_rate_q,q_span,answer,a_verbatim,match_rate_a,a_span
10,podcasts-audio/5/9/show_59h8QSNa9n0SsLDAdH2GAA/4eNkaKFM0Ex4dbiExiuHlX,What are the advantages of staying or returning to Real Madrid?,What are the advantages of staying or returning to Real Madrid,1.0,"(3801, 3862)",What is clear is I want to play and not be sat on the bench.,What is clear is I want to play and not be sat on the bench,1.000000,"(3865, 3923)"
21,podcasts-audio/3/X/show_3XVVceNECAwk8xvSSYdgIO/7itwLRPH0t1ai26BTdztAA,Do you have any comments or advice regarding future pre-med applicants who either come from business or marketing or just kind of is transitioning right now?,do you have any comments or advice regarding future pre-med applicants who either come from business or marketing or just kind of is transitioning right now,1.0,"(47609, 47764)","For anyone who like thinks that they want to quit their job and go into a new career you mean, I would say really take time to reflect and decide, see why you want to do it. Are you just bored wit...","i would say really take time to reflect and decide, see why you want to do it. Are you just bored with your current job? Are you trying different jobs? Is it like a longer term life issue where yo...",0.963855,"(47867, 50396)"


--------------------------------------